# 4. Ekstraksi Data NO₂ — Kecamatan Kedungpring

**Mata Kuliah:** Proyek Sains Data — Semester 5
**Wilayah:** Kecamatan Kedungpring, Kabupaten Lamongan
**Polutan:** NO₂ (Nitrogen Dioksida)
**Rentang Waktu:** 31 Agustus 2025 — 31 Agustus 2026

## 2. Import Pustaka

- `openeo` — client Python untuk mengakses backend openEO Copernicus Data Space.
- `xarray` — membaca file NetCDF hasil ekstraksi dan mengubahnya menjadi tabel.
- `pandas` — manipulasi dan penyimpanan data dalam format CSV.
- `os` — memastikan folder penyimpanan tersedia sebelum data ditulis.

In [ ]:
import os
import openeo
import xarray as xr
import pandas as pd

NC_DIR = "../data/nc/"
CSV_DIR = "../data/csv/"
os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Folder output siap:")
print(" -", os.path.abspath(NC_DIR))
print(" -", os.path.abspath(CSV_DIR))

Folder output siap:
 - d:\Semester 5\Proyek sain data\PSD\data\nc
 - d:\Semester 5\Proyek sain data\PSD\data\csv


## 3. Koneksi & Otentikasi ke Copernicus Data Space

Sama seperti tahap ekstraksi sebelumnya, koneksi ke server openEO memerlukan otentikasi
**OIDC Device Code Flow**:

1. Saat sel di bawah dijalankan, akan muncul tautan otentikasi di output.
2. Buka tautan tersebut di browser, lalu login menggunakan akun Copernicus Data Space
   Ecosystem (CDSE).
3. Setelah berhasil login, proses di notebook akan otomatis melanjutkan tanpa perlu
   memasukkan token atau password apa pun secara manual.

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

print("Terhubung sebagai:", connection.describe_account())

Authenticated using refresh token.
Terhubung sebagai: {'info': {'oidc_userinfo': {'email': 'triswanti1395@gmail.com', 'email_verified': True, 'family_name': "Jannatul Ma'wa", 'given_name': 'Triswanti', 'name': "Triswanti Jannatul Ma'wa", 'preferred_username': 'triswanti1395@gmail.com', 'sub': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}}, 'name': "Triswanti Jannatul Ma'wa", 'user_id': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}


## 4. Area of Interest (AOI): Kecamatan Kedungpring

Koordinat batas berikut dikonversi dari data resmi *Laporan Kinerja Instansi Pemerintah
(LKjIP) Kecamatan Kedungpring 2024*, dengan rentang koordinat asli
**112°10′01″–112°13′28″ BT** dan **07°08′19″–07°12′27″ LS**.

| Atribut | Nilai |
| ------- | ----- |
| Luas wilayah | 84,54 km² |
| Jumlah desa | 23 desa |
| Batas Utara | Kecamatan Babat |
| Batas Timur | Kecamatan Sugio |
| Batas Selatan | Kecamatan Ngimbang |
| Batas Barat | Kecamatan Modo |

In [ ]:
# Bounding box untuk parameter spatial_extent pada load_collection
KEDUNGPRING_BBOX = {
    "west": 112.1669,
    "south": -7.2075,
    "east": 112.2244,
    "north": -7.1386,
}

# Polygon (GeoJSON) untuk parameter geometries pada aggregate_spatial
KEDUNGPRING_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [112.1669, -7.1386],
        [112.2244, -7.1386],
        [112.2244, -7.2075],
        [112.1669, -7.2075],
        [112.1669, -7.1386],
    ]]
}

print("AOI Kecamatan Kedungpring:", KEDUNGPRING_BBOX)

AOI Kecamatan Kedungpring: {'west': 112.1669, 'south': -7.2075, 'east': 112.2244, 'north': -7.1386}


## 5. Memuat Data (Load Collection)

Data dimuat dari koleksi `SENTINEL_5P_L2`, dibatasi pada band **NO2** saja (sesuai
kebutuhan tahap ekstraksi fitur berikutnya yang berfokus pada polutan NO₂), dengan
rentang waktu **31 Agustus 2025 – 31 Agustus 2026**.

In [ ]:
TEMPORAL_EXTENT = ["2025-08-31", "2026-08-31"]

datacube = connection.load_collection(
    "SENTINEL_5P_L2",
    spatial_extent=KEDUNGPRING_BBOX,
    temporal_extent=TEMPORAL_EXTENT,
    bands=["NO2"],
)

print("Datacube NO2 berhasil dimuat untuk rentang waktu:", TEMPORAL_EXTENT)

Datacube NO2 berhasil dimuat untuk rentang waktu: ['2025-08-31', '2026-08-31']


## 6. Agregasi Temporal & Spasial

Dua langkah agregasi dilakukan agar data satelit (yang aslinya berupa citra raster)
berubah menjadi **satu nilai per hari** yang mewakili seluruh wilayah Kecamatan
Kedungpring:

1. **Agregasi temporal** (`aggregate_temporal_period`) — menghitung rata-rata harian,
   karena dalam satu hari bisa ada lebih dari satu lintasan satelit.
2. **Agregasi spasial** (`aggregate_spatial`) — menghitung rata-rata nilai piksel yang
   berada di dalam polygon Kedungpring, menghasilkan satu angka representatif per hari.

In [ ]:
daily_cube = datacube.aggregate_temporal_period(period="day", reducer="mean")
spatial_result = daily_cube.aggregate_spatial(geometries=KEDUNGPRING_POLYGON, reducer="mean")

print("Agregasi temporal (harian) dan spasial (rata-rata dalam AOI) selesai didefinisikan.")

Agregasi temporal (harian) dan spasial (rata-rata dalam AOI) selesai didefinisikan.


## 7. Mengunduh Hasil & Konversi ke CSV

Hasil akhir diproses di server openEO lalu diunduh sebagai file **NetCDF (`.nc`)**.
Proses ini bisa memakan waktu beberapa menit tergantung beban server dan panjang
rentang waktu yang diminta (di sini: 1 tahun penuh).

Setelah file `.nc` tersedia, data dibaca dengan `xarray` dan dikonversi menjadi
**CSV** dengan dua kolom: `date` dan `NO2` — format ini dibuat agar langsung kompatibel
dengan tahap pemrosesan berikutnya (deteksi outlier, imputasi, dan ekstraksi fitur
TSFEL).

In [ ]:
nc_path = os.path.join(NC_DIR, "no2_kedungpring.nc")
print("Mengunduh hasil ke:", nc_path, "... (mohon tunggu)")
spatial_result.download(nc_path, format="netCDF")
print("Unduhan NetCDF selesai.")

Mengunduh hasil ke: ../data/nc/no2_kedungpring.nc ... (mohon tunggu)
Unduhan NetCDF selesai.


In [ ]:
ds = xr.open_dataset(nc_path)
df = ds.to_dataframe().reset_index()

print("Kolom hasil NetCDF:")
print(df.columns.tolist())

# Mencari kolom yang berisi data waktu/tanggal
date_col = None

for c in df.columns:
    try:
        converted = pd.to_datetime(df[c], errors="coerce")
        if converted.notna().sum() > 0:
            date_col = c
            break
    except:
        pass

if date_col is None:
    raise ValueError(
        "Kolom tanggal/waktu tidak ditemukan. "
        "Silakan cek daftar kolom yang ditampilkan di atas."
    )

# Mencari kolom nilai NO2
value_col = None

# Prioritas kolom yang namanya mengandung NO2
for c in df.columns:
    if "no2" in c.lower():
        value_col = c
        break

# Jika nama NO2 tidak ditemukan, gunakan kolom numerik selain tanggal
if value_col is None:
    numeric_cols = df.select_dtypes(include="number").columns.tolist()

    if date_col in numeric_cols:
        numeric_cols.remove(date_col)

    if len(numeric_cols) > 0:
        value_col = numeric_cols[-1]

if value_col is None:
    raise ValueError(
        "Kolom nilai NO2 tidak ditemukan. "
        "Silakan cek daftar kolom yang ditampilkan di atas."
    )

print("Kolom tanggal :", date_col)
print("Kolom NO2     :", value_col)

# Ambil hanya tanggal dan nilai NO2
df = df[[date_col, value_col]].copy()

# Rename menjadi format yang dibutuhkan
df = df.rename(columns={
    date_col: "date",
    value_col: "NO2"
})

# Konversi tanggal
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Hapus baris yang tanggalnya tidak valid
df = df.dropna(subset=["date"])

# Ubah menjadi format tanggal saja
df["date"] = df["date"].dt.date

# Simpan CSV
csv_path = os.path.join(CSV_DIR, "NO2-Kedungpring.csv")
df.to_csv(csv_path, index=False)

print(f"CSV berhasil disimpan: {csv_path} ({len(df)} baris)")

df.head()

Kolom hasil NetCDF:
['t', 'feature', 'NO2', 'lat', 'lon', 'feature_names']
Kolom tanggal : t
Kolom NO2     : NO2
CSV berhasil disimpan: ../data/csv/NO2-Kedungpring.csv (169 baris)


,date,NO2
0,2025-08-31,0.000036
1,2025-09-01,0.000020
2,2025-09-03,0.000013
3,2025-09-05,0.000029
4,2025-09-11,0.000030


## 8. Verifikasi Hasil Ekstraksi

Sebelum lanjut ke tahap preprocessing (deteksi outlier & imputasi missing value), kita
periksa dulu ringkasan data yang berhasil diekstrak — termasuk jumlah baris dan
proporsi nilai kosong (NaN) yang akan ditangani pada tahap berikutnya.

In [ ]:
print(f"Total baris (hari) dalam rentang waktu ekstraksi : {len(df)}")
print(f"Jumlah nilai NO2 terisi (valid)                    : {df['NO2'].notna().sum()}")
print(f"Jumlah nilai NO2 kosong (NaN)                       : {df['NO2'].isna().sum()}")
print(f"Rentang tanggal data                                : {df['date'].min()} s.d. {df['date'].max()}")

Total baris (hari) dalam rentang waktu ekstraksi : 169
Jumlah nilai NO2 terisi (valid)                    : 169
Jumlah nilai NO2 kosong (NaN)                       : 0
Rentang tanggal data                                : 2025-08-31 s.d. 2026-08-30


## 9. Ringkasan Tahap 1

| Item | Nilai |
| ---- | ----- |
| Wilayah | Kecamatan Kedungpring, Kabupaten Lamongan |
| Polutan | NO₂ |
| Rentang waktu | 31 Agustus 2025 – 31 Agustus 2026 |
| Sumber data | Sentinel-5P L2, via openEO Copernicus Data Space |
| Output | `../data/nc/no2_kedungpring.nc`, `../data/csv/NO2-Kedungpring.csv` |

Data mentah pada tahap ini **masih mengandung missing value** (akibat tutupan awan,
jadwal orbit satelit, atau validasi kualitas data) — hal ini normal dan akan ditangani
secara eksplisit pada tahap berikutnya.
